In [ ]:
%run ../tardis_eda.ipynb

# Tardis map : initialisation

- modules calls
- define exit values

In [ ]:
import pandas as pd
import folium as flm
from datetime import date, time
from tqdm import trange
from os import getcwd, chdir

cwd = getcwd()
if cwd.endswith("/bonus"):
    chdir(cwd + "/..")

from tardis_model import question_the_model, TRAIN_SERVICE, SEC_IN_MIN


EPITECH_SUCCESS = 0
EPITECH_FAILURE = 84

GET_NBR_ROWS = lambda df: df.shape[0]
MONTH_IN_YEAR = 12
MIN_IN_H = 60
SEC_IN_MIN = 60
SEC_IN_H = MIN_IN_H * SEC_IN_MIN
H_IN_DAY = 24

FOREIGN_STATIONS = (
    "FRANCFORT",
    "STUTTGART",
    "ZURICH",
    "LAUSANNE",
    "ITALIE",
    "BARCELONA",
    "MADRID",
)

# MAP_CENTER = (48.836536, 2.35632)
MAP_CENTER = (46.92475, 2.51240)
MAP_ZOOM = 6.2

COLOR_STEP = 5
STATION_ICON = "train"
LINE_WEIGHT = 2.5

#### Map creation

no argument

return the folium Map object centered on `MAP_CENTER` and zoom at `MAP_ZOOM`

In [70]:
def init_map() -> flm.Map:
    map = flm.Map(location=MAP_CENTER, zoom_start=MAP_ZOOM, tiles="OpenStreetMap")
    tl = flm.TileLayer(
        tiles="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
        attr="&copy; <a href='https://www.openstreetmap.org/copyright'>OpenStreetMap</a> contributors",
    )
    tl.options["Referrer-Policy"] = "no-referrer-when-downgrade"
    tl.add_to(map)
    return map

In [71]:
def get_field_value(ser: pd.Series, wanted_type) -> any:
    val = 0
    # val = ser.values[0]
    for e in ser:
        val = e
        break
    if wanted_type == date and type(val) == pd.Timestamp:
        val = val.date()
    pd_type = wanted_type
    if wanted_type == int:
        pd_type = float
    if type(val) == pd_type:
        if wanted_type == int:
            return int(val)
        return val
    if wanted_type == str:
        return ""
    return None

#### Get the stations names

arguments :
- csv (panda DataFrame), the dataframe

return the list of stations names (list of str)\
return None in case of error

In [72]:
def get_stations_data() -> dict[str : tuple[float, float]]:
    try:
        csv = pd.read_csv("bonus/stations_coords.csv", sep=",")
    except:
        return None
    for c in ("Latitude", "Longitude"):
        csv[c] = pd.to_numeric(csv[c], errors="coerce")
    data = {}
    for i in range(GET_NBR_ROWS(csv)):
        name = get_field_value(csv.iloc[[i]]["Name"], str)
        lat = get_field_value(csv.iloc[[i]]["Latitude"], float)
        long = get_field_value(csv.iloc[[i]]["Longitude"], float)
        if (
            type(name) == str
            and type(lat) in (int, float)
            and type(long) in (int, float)
        ):
            data[name] = (lat, long)
    return data

#### Get color for objects + Convertion time (in minutes) to time (formatted as (j) hh:mm:ss)

color guide :
- <span style="color: green">green</span> = average delay <= 5 minutes
- <span style="color: blue">blue</span> = 5 minutes < average delay <= 10 minutes
- <span style="color: yellow">yellow</span> = 10 minutes < average delay <= 15 minutes
- <span style="color: orange">orange</span> = 15 minutes < average delay <= 20 minutes
- <span style="color: red">red</span> = 20 minutes < average delay <= 30 minutes
- <span style="color: purple">purple</span> = average delay > 30 minutes
- <span style="color: gray">gray</span> = no data

In [74]:
def get_color(delay: float) -> str:
    if delay < 0:
        return "gray"
    colors = ("green", "blue", "yellow", "orange", "red", "red", "purple")
    i = 0
    while delay >= COLOR_STEP and i < (len(colors) - 1):
        delay -= COLOR_STEP
        i += 1
    return colors[i]


def min_to_time(nbr: int) -> str:
    if nbr < 0:
        return "-" + min_to_time(-nbr)
    n_sec = int(round(nbr * SEC_IN_MIN, 0))
    d = n_sec // (SEC_IN_H * H_IN_DAY)
    n_sec = n_sec % (SEC_IN_H * H_IN_DAY)
    h = n_sec // SEC_IN_H
    n_sec = n_sec % SEC_IN_H
    m = n_sec // SEC_IN_MIN
    sec = n_sec % SEC_IN_MIN
    text = f"{d}j" if d > 0 else ""
    return text + f"{h:02}:{m:02}:{sec:02}"

#### Draw stations one the map :

arguments :
- map (folium Map), the map object
- stations ({str : (float, float)} dict), the stations data

draw the stations markers on the map\
doesn't return anything

In [75]:
def time_obj_to_minutes(t: time) -> float:
    return t.hour + t.minute + (t.second / SEC_IN_MIN)


def get_train_service(s1: str, s2: str) -> TRAIN_SERVICE:
    if s1.upper() in FOREIGN_STATIONS or s2.upper() in FOREIGN_STATIONS:
        return TRAIN_SERVICE.International
    return TRAIN_SERVICE.National


def get_station_weight(name: str, stations: dict[str : tuple[float, float]]) -> float:
    today = date.today()
    s, n = 0, 0
    for dep in stations:
        tmp = question_the_model(dep, name, today, get_train_service(dep, name))
        if not (tmp is None):
            s += time_obj_to_minutes(tmp)
            n += 1
    if n <= 0:
        return -1
    return s / n


def draw_stations(map: flm.Map, stations: dict[str : tuple[float, float]]) -> None:
    print("Draw stations")
    names = list(stations.keys())
    for i in trange(0, len(names), 1):
        k = names[i]
        delay = get_station_weight(k, stations)
        c = get_color(delay)
        if c == "yellow":
            c = "beige"
        if delay < 0:
            popup = f"{k} : NaN"
        else:
            popup = f"{k} : {min_to_time(delay)}"
        flm.Marker(
            location=list(stations[k]),
            icon=flm.Icon(color=c, prefix="fa", icon=STATION_ICON),
            popup=flm.Popup(popup, max_width=None),
        ).add_to(map)

#### Draw routes one the map :

arguments :
- map (folium Map), the map object
- stations ({str : (float, float)} dict), the stations data

draw the route polylines on the map\
doesn't return anything

In [76]:
def draw_lines(map: flm.Map, stations: dict[str : tuple[float, float]]) -> None:
    print("Draw lines :")
    names = list(stations.keys())
    l = len(names)
    for i in trange(l - 1):
        k1 = names[i]
        for j in range(i + 1, l):
            k2 = names[j]
            if k2 == k1:
                continue
            delay1 = question_the_model(k1, k2, date.today(), get_train_service(k1, k2))
            delay2 = question_the_model(k2, k1, date.today(), get_train_service(k1, k2))
            if delay1 is None or delay2 is None:
                delay = -1
            else:
                delay = time_obj_to_minutes(delay1) + time_obj_to_minutes(delay2)
            if delay < 0:
                continue
            c = get_color(delay)
            popup = f"{k1} ↔ {k2} : {min_to_time(delay)}"
            flm.PolyLine(
                [list(stations[k1]), list(stations[k2])],
                color=c,
                weight=LINE_WEIGHT,
                tooltip=popup,
            ).add_to(map)
    print()

In [77]:
def draw_map(stations: bool = True, lines: bool = False) -> flm.Map:
    if stations or lines:
        data = get_stations_data()
        if data is None:
            return None
    map = init_map()
    if stations:
        draw_stations(map, data)
    if lines:
        draw_lines(map, data)
    return map


def main_map() -> int:
    map = draw_map(stations=True, lines=True)
    if map is None:
        chdir(getcwd() + "/bonus")
        return EPITECH_FAILURE
    print("Save the map")
    map.save("maps/map_today.html")
    print("Finished !")
    chdir(getcwd() + "/bonus")
    return EPITECH_SUCCESS


if __name__ == "__main__":
    main_map()

Draw stations


100%|██████████| 59/59 [00:08<00:00,  6.57it/s]


Draw lines :


100%|██████████| 58/58 [00:08<00:00,  6.67it/s]


Save the map
Finished !
